# Мониторинг обучения диффузионной модели

Ячейки можно перезапускать в любой момент, пока идет обучение, они просто перечитывают файлы из папки запуска.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

RUN_DIR = Path('/home/CCP/semenchuk.aa/DiffusionLearn/runs/cat64_crop_fit')


def read_csv(path):
    # читает csv в словарь {колонка: np.array}
    path = Path(path)
    if not path.exists():
        return {}
    with open(path) as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}
    return {k: np.array([float(r[k]) for r in rows]) for k in rows[0]}

## Метрики

In [ ]:
log = read_csv(RUN_DIR / 'log.csv')
nn = read_csv(RUN_DIR / 'nn_stats.csv')

fig, axs = plt.subplots(2, 2, figsize=(16, 10))
for ax in axs.flat:
    ax.grid(alpha=0.5, zorder=1, which='both')

# 1. общий лосс
axs[0, 0].semilogy(log['step'], log['loss'], zorder=2)
axs[0, 0].set_title('лосс, среднее по t')

# 2. лосс по интервалам t
bins = ['loss_t0-0.2', 'loss_t0.2-0.4', 'loss_t0.4-0.6', 'loss_t0.6-0.8', 'loss_t0.8-1']
for b in bins:
    axs[0, 1].semilogy(log['step'], log[b], label='t ' + b.replace('loss_t', ''), zorder=2)
axs[0, 1].set_title('лосс по интервалам t')
axs[0, 1].legend()

# 3. проверка на запоминание
if nn:
    axs[1, 0].plot(nn['step'], nn['nn_median'], 'o-', label='сэмплы, медиана', zorder=2)
    axs[1, 0].plot(nn['step'], nn['nn_min'], 'o-', label='сэмплы, минимум', zorder=2)
    axs[1, 0].axhline(nn['train_nn_median'][0], color='k', ls='--', label='обучающие фото между собой', zorder=2)
    axs[1, 0].legend()
axs[1, 0].set_title('расстояние до ближайшего обучающего фото')

# 4. норма градиента и learning rate
axs[1, 1].plot(log['step'], log['grad_norm'], color='C0', zorder=2)
axs[1, 1].set_ylabel('grad norm', color='C0')
ax2 = axs[1, 1].twinx()
ax2.plot(log['step'], log['lr'], color='C1', zorder=2)
ax2.set_ylabel('lr', color='C1')
axs[1, 1].set_title('норма градиента и learning rate')

for ax in axs[1]:
    ax.set_xlabel('step')

plt.tight_layout()
plt.show()

print(f"шаг {int(log['step'][-1])}, лосс {log['loss'][-1]:.4f}, скорость {log['it_per_s'][-1]:.1f} it/s")

## Последние сэмплы

In [ ]:
samples = sorted((RUN_DIR / 'samples').glob('step_???????.png'))

if samples:
    last = samples[-1]
    pairs = last.with_name(last.stem + '_nn.png')

    fig, axs = plt.subplots(1, 2, figsize=(20, 10))
    axs[0].imshow(Image.open(last))
    axs[0].set_title(f'{last.stem}, один и тот же стартовый шум на всех шагах')
    axs[1].imshow(Image.open(pairs))
    axs[1].set_title('пары: сэмпл и ближайшее обучающее фото')
    for ax in axs:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('сэмплов пока нет, первые появятся на шаге 2500')

## Как менялись одни и те же 8 кошек по ходу обучения

In [ ]:
# первая строка каждой сетки: 8 сэмплов из одного и того же шума
ROW = 2 + 128 + 2   # отступ + картинка 64x64, увеличенная в 2 раза + отступ

rows = [np.asarray(Image.open(p))[:ROW] for p in samples]
if rows:
    fig, ax = plt.subplots(figsize=(12, 1.6 * len(rows)))
    ax.imshow(np.concatenate(rows, axis=0))
    ax.set_yticks([ROW * (i + 0.5) for i in range(len(rows))])
    ax.set_yticklabels([p.stem.replace('step_', '') for p in samples])
    ax.set_xticks([])
    ax.set_title('строка = шаг обучения')
    plt.show()

---

In [ ]:
import io
import math
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from torchvision.utils import save_image

from models import model_from_checkpoint
from train import sde_from_cfg

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
_cache = {}


def load_model(ckpt_path):
    """Модель, расписание шума, шаг и размер картинки из чекпоинта.
    Если файл не менялся с прошлого вызова, берем из кэша."""
    ckpt_path = Path(ckpt_path)
    key = (str(ckpt_path), ckpt_path.stat().st_mtime)
    if _cache.get('key') != key:
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        _cache.update(key=key, model=model_from_checkpoint(ck).to(DEVICE).eval(),
                      sde=sde_from_cfg(ck['args']), step=ck['step'], size=ck['args']['img_size'])
        print(f"загружен {ckpt_path}, шаг {ck['step']}, модель {ck.get('model_cfg')}")
    return _cache['model'], _cache['sde'], _cache['step'], _cache['size']


def to_np(img):
    """(3, H, W) в [-1, 1] -> (H, W, 3) в [0, 1] для imshow."""
    return ((img.permute(1, 2, 0).float().numpy() + 1) / 2).clip(0, 1)


@torch.no_grad()
def generate(model, sde, n, size, seed=0, method='sde', solver='heun', steps=1000, rho=None):
    """
    Генерирует n картинок одним батчем.
      method   : 'sde' (Эйлер-Маруяма) или 'ode' (вероятностное ОДУ)
      solver   : для ode, 'heun' или 'euler'
      rho      : сгущение сетки по t к нулю; по умолчанию 1 для sde и 2 для ode
    Возвращает final (n, 3, size, size) в [-1, 1] на CPU и history,
    список словарей на каждый шаг с t, sigma, x (n, 3, size, size) и x0 (догадка модели о финальной картинке).
    Кадры лежат на CPU в float16: при 1000 шагах и 16 картинках это около 800 МБ.
    """
    rho = rho if rho is not None else (1.0 if method == 'sde' else 2.0)
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 3, size, size, generator=g, device=DEVICE)
    u = np.linspace(1.0, 0.0, steps + 1)
    ts = sde.t_min + (1.0 - sde.t_min) * u ** rho
    T = lambda t: torch.tensor(float(t), device=DEVICE)

    def eps_at(x, t):
        tt = torch.full((n,), float(t), device=DEVICE)
        return sde.to_eps(model(x, sde.c(tt)).float(), x, tt)

    def x0_hat(x, t, e):
        return (x - sde.sigma(T(t)) * e) / sde.m(T(t))

    def drift(x, t, e, k):  # k = 1 для ОДУ, k = 2 для SDE
        return 0.5 * sde.beta(T(t)) * (x - k * e / sde.sigma(T(t)))

    history = []
    for i in range(steps + 1):
        t = ts[i]
        e = eps_at(x, t)
        history.append(dict(t=float(t), sigma=sde.sigma(T(t)).item(),
                            x=x.clamp(-1, 1).half().cpu(), x0=x0_hat(x, t, e).clamp(-1, 1).half().cpu()))
        if i == steps:
            break
        h = t - ts[i + 1]
        if method == 'sde':
            x = x + h * drift(x, t, e, 2)
            if i < steps - 1:
                x = x + math.sqrt(sde.beta(T(t)).item() * h) * torch.randn(x.shape, generator=g, device=DEVICE)
        elif solver == 'euler':
            x = x + h * drift(x, t, e, 1)
        else:  # heun
            d = drift(x, t, e, 1)
            x_e = x + h * d
            x = x + 0.5 * h * (d + drift(x_e, ts[i + 1], eps_at(x_e, ts[i + 1]), 1))
    return x0_hat(x, ts[-1], e).clamp(-1, 1).cpu(), history

In [ ]:
# ------------------------------------------------------------- параметры
RUN = 'runs/cat64_crop_fit'
CKPT = 'last.pt'                 # или 'ema_0060000.pt'
# CKPT = 'ema_0020000.pt'
SEED = 67
BATCH = 16
METHOD = 'sde'                   # 'sde' или 'ode'
SOLVER = 'heun'                  # для ode: 'heun' или 'euler'
STEPS = 1000 if METHOD == 'sde' else 50
N_FRAMES = 12                    # кадров на рисунке в следующей ячейке
COLS = 8                         # картинок в строке на рисунке

# ------------------------------------------------------------- генерация
model, sde, step, S = load_model(f'{RUN}/{CKPT}')
final, history = generate(model, sde, BATCH, S, seed=SEED, method=METHOD, solver=SOLVER,
                          steps=STEPS)

# каждая картинка в отдельный файл, 64x64 без увеличения
name = f'step{step:07d}_{METHOD}' + (f'-{SOLVER}' if METHOD == 'ode' else '') + f'{STEPS}_seed{SEED}'
out_dir = Path(RUN) / 'generated' / name
out_dir.mkdir(parents=True, exist_ok=True)
for i, img in enumerate(final):
    save_image((img + 1) / 2, out_dir / f'{i:03d}.png')
print(f'{BATCH} картинок сохранено в {out_dir}')

# ------------------------------------------------------------- рисунок
cols = min(COLS, BATCH)
rows = math.ceil(BATCH / cols)
fig, axs = plt.subplots(rows, cols, figsize=(2.3 * cols, 2.5 * rows), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for i, img in enumerate(final):
    ax = axs.flat[i]
    ax.imshow(to_np(img), interpolation='bilinear')
    ax.set_title(str(i), fontsize=10)
plt.suptitle(f'{RUN}, шаг {step}, SEED = {SEED}, {METHOD}' + (f' ({SOLVER})' if METHOD == 'ode' else ''), fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
IDX = 0       # номер картинки из батча, он подписан над ней на рисунке выше
COLS_H = 6

frames = [history[i] for i in np.linspace(0, len(history) - 1, N_FRAMES).round().astype(int)]
rows = math.ceil(len(frames) / COLS_H)
fig, axs = plt.subplots(2 * rows, COLS_H, figsize=(3.2 * COLS_H, 3.6 * 2 * rows))
for ax in axs.flat:
    ax.axis('off')

for j, fr in enumerate(frames):
    r, c = divmod(j, COLS_H)
    top, bottom = axs[2 * r, c], axs[2 * r + 1, c]

    top.imshow(to_np(fr['x'][IDX]), interpolation='bilinear')
    top.set_title(f"$x_t$,  t = {fr['t']:.3f}\n$\\sigma$ = {fr['sigma']:.3f}", fontsize=11)

    bottom.imshow(to_np(fr['x0'][IDX]), interpolation='bilinear')
    bottom.set_title('что видит модель, $\\hat x_0$', fontsize=11)

plt.suptitle(f'картинка {IDX}, SEED = {SEED}, {METHOD}' + (f' ({SOLVER})' if METHOD == 'ode' else ''), fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
IDX = 11       # номер картинки из батча, он подписан над ней на рисунке выше
WIDTH = 384    # сторона картинки на экране, в пикселях


def to_png(img):
    """(3, H, W) в [-1, 1] -> байты png для widgets.Image."""
    buf = io.BytesIO()
    Image.fromarray((to_np(img) * 255).round().astype(np.uint8)).save(buf, 'png')
    return buf.getvalue()


# кадры кодируем заранее, поэтому другую картинку можно выбрать без новой генерации: поменять IDX и перезапустить ячейку
x_png = [to_png(fr['x'][IDX]) for fr in history]
x0_png = [to_png(fr['x0'][IDX]) for fr in history]
last = len(history) - 1

box = widgets.Layout(width=f'{WIDTH}px', height=f'{WIDTH}px')
img_x = widgets.Image(value=x_png[0], format='png', layout=box)
img_x0 = widgets.Image(value=x0_png[0], format='png', layout=box)
info = widgets.HTML()

# Play двигает ползунок сам: 50 шагов за 10 с, 1000 шагов за 30 с, быстрее 30 мс на кадр не идем
play = widgets.Play(min=0, max=last, interval=max(30, 10_000 // len(history)), description='запуск')
slider = widgets.IntSlider(min=0, max=last, description='шаг', layout=widgets.Layout(width=f'{2 * WIDTH - 80}px'))
widgets.jslink((play, 'value'), (slider, 'value'))


def show_frame(i):
    img_x.value, img_x0.value = x_png[i], x0_png[i]
    fr = history[i]
    info.value = f"<b>картинка {IDX}, SEED = {SEED}, {METHOD}</b>: шаг {i} из {last}, t = {fr['t']:.3f}, &sigma; = {fr['sigma']:.3f}"


slider.observe(lambda change: show_frame(change['new']), names='value')
show_frame(0)

display(widgets.VBox([
    info,
    widgets.HBox([widgets.VBox([widgets.HTML('x<sub>t</sub>'), img_x]),
                  widgets.VBox([widgets.HTML('что видит модель, x&#770;<sub>0</sub>'), img_x0])]),
    widgets.HBox([play, slider]),
]))

---

In [ ]:
# ------------------------------------------------------------- запоминание: сравнение запусков
# MEM  = Path('/home/CCP/semenchuk.aa/DiffusionLearn/runs/memorization')
MEM = Path('/home/CCP/semenchuk.aa/DiffusionLearn/runs/memorization2')
# RUNS = ['cat64_eps', 'cat64_uniq']
RUNS = ['cat64_uniq', 'cat64_small_gn8']
TAUS = [0.5, 0.7]          # порог ratio = d1 / d2: ниже него считаем сэмпл копией

data = {name: np.load(MEM / f'{name}.npz') for name in RUNS if (MEM / f'{name}.npz').exists()}

fig, axs = plt.subplots(2, 2, figsize=(16, 11))
for ax in axs.flat:
    ax.grid(alpha=0.5, zorder=1)

for k, (name, z) in enumerate(data.items()):
    c = f'C{k}'
    steps, r, br = z['steps'], z['ratio'], z['base_ratio']

    # 1, 2. доля копий при двух порогах; пунктир = доля среди настоящих фото
    for j, tau in enumerate(TAUS):
        ax = axs[0, j]
        ax.plot(steps, (r < tau).mean(1) * 100, 'o-', color=c, label=f'{name}, сэмплы', zorder=2)
        ax.axhline((br < tau).mean() * 100, color=c, ls='--', label=f'{name}, настоящие фото', zorder=2)
        ax.set_title(f'доля сэмплов с ratio < {tau}, %')
        ax.set_xlabel('step')

    # 3. медиана расстояния до ближайшего фото
    axs[1, 0].plot(steps, np.median(z['d1'], 1), 'o-', color=c, label=f'{name}, сэмплы', zorder=2)
    axs[1, 0].axhline(np.median(z['base_d1']), color=c, ls='--', label=f'{name}, настоящие фото', zorder=2)

    # 4. распределение ratio на последнем чекпоинте и у настоящих фото
    bins = np.linspace(0, 1.05, 43)
    axs[1, 1].hist(r[-1], bins=bins, density=True, histtype='step', lw=2, color=c,
                   label=f'{name}, сэмплы, шаг {steps[-1]}', zorder=2)
    axs[1, 1].hist(br, bins=bins, density=True, histtype='step', lw=1.5, ls='--', color=c,
                   label=f'{name}, настоящие фото', zorder=2)

axs[1, 0].set_title('медиана расстояния до ближайшего фото (с учетом кропов)')
axs[1, 0].set_xlabel('step')
axs[1, 1].set_title('распределение ratio = d1 / d2')
axs[1, 1].set_xlabel('ratio')
for t in TAUS:
    axs[1, 1].axvline(t, color='gray', lw=1, zorder=1)
for ax in axs.flat:
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# таблица по шагам
for name, z in data.items():
    print(f'\n{name}: настоящие фото ratio<0.5 = {(z["base_ratio"] < 0.5).mean() * 100:.1f}%')
    for s, r in zip(z['steps'], z['ratio']):
        print(f'  шаг {s:7d}   ratio<0.5 {(r < 0.5).mean() * 100:5.1f}%   ratio<0.7 {(r < 0.7).mean() * 100:5.1f}%')

# 16 самых подозрительных сэмплов на последнем чекпоинте: слева сэмпл, справа ближайший кроп фото
for name, z in data.items():
    p = MEM / f'{name}_step{int(z["steps"][-1]):07d}_top16.png'
    if p.exists():
        fig, ax = plt.subplots(figsize=(14, 4.5))
        ax.imshow(Image.open(p))
        ax.set_title(f'{name}: 16 сэмплов с наименьшим ratio (пары: сэмпл, ближайшее фото)')
        ax.axis('off')
        plt.show()